In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# download the names.txt file from github
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87Saving 'names.txt.12'
87names.txt.12         100% [=============================>]  100.93K    --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/karpathy/makemore/master/names.txt]
87names.txt.12         100% [=============================>]  100.93K    --.-KB/s87[Files: 1  Bytes: 100.93K [184.]8

In [3]:
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(x) for x in words))
print(words[:5])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia']


In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['$'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(stoi)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '$'}
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '$': 0}
27


In [5]:
# Building Dataset
block_size = 3
def build_dataset(words):
    X, y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '$':
            ix = stoi[ch]
            X.append(context)
            y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    y = torch.tensor(y)
    print(X.shape, y.shape)
    return X, y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

X_train, y_train = build_dataset(words[:n1])
X_valid, y_valid = build_dataset(words[n1:n2])
X_test, y_test = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
def cmp(s, dt, t): # (name, our_grads, pytorch_grads)
    ex = torch.all(dt == t.grad).item()
    approx = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(approx):5s} | maxdiff: {maxdiff}')

In [7]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((vocab_size, n_embd),             generator=g)
# Layer 1
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3)/((n_embd * block_size) ** 0.5)
B1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
B2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm Parameters
bn_gain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bn_bias = torch.randn((1, n_hidden)) * 0.1

# Note: initializating many of these parameters in non-standard ways because sometimes initializating with e.g. all zeros could mask an incorrect implementation of the backward pass.

parameters = [C, W1, B1, W2, B2, bn_gain, bn_bias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

4137


In [8]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
Xb, yb = X_train[ix], y_train[ix] # batch X, y
Xb.shape, yb.shape

(torch.Size([32, 3]), torch.Size([32]))

In [9]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb]
emb_cat = emb.view(emb.shape[0], -1)

# Linear Layer 1
hpre_bn = emb_cat @ W1 + B1

# BatchNorm Layer
bn_mean_i = 1/n*hpre_bn.sum(0, keepdim=True) # hpre_bn.mean(0, keepdim=True)[:, :5]
bn_diff = hpre_bn - bn_mean_i
bn_diff_2 = bn_diff ** 2
bn_var = 1/(n-1)*(bn_diff_2).sum(0, keepdim=True) # this is a batch~sample so its n-1 # note Bessel's correction (dividing by n-1, not n)
bn_var_inv = (bn_var + 1e-5)**(-0.5) # Batch Normalization formula -> (X - mean) / sqrt(variance + epsilon), this is denominator
bn_raw = bn_diff * bn_var_inv # this is numerator
hpre_act = bn_gain * bn_raw + bn_bias

# Non-Linearity
h = torch.tanh(hpre_act)

# Linear Layer 2
logits = h @ W2 + B2

# Cross Entropy Loss (same as F.cross_entropy(logits, yb))
logits_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logits_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** (-1) # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
log_probs = probs.log()
loss = -log_probs[range(n), yb].mean()

for p in parameters:
    p.grad = None

for t in [log_probs, probs, counts, counts_sum, counts_sum_inv, norm_logits, logits_maxes, logits, h, hpre_act, bn_raw, bn_var_inv, bn_var, bn_diff_2, bn_diff, hpre_bn, bn_mean_i, emb_cat, emb]:
    t.retain_grad()

loss.backward()
loss

tensor(3.3381, grad_fn=<NegBackward0>)

In [ ]:
# d_log_probs
# loss = -(1/3)*a + -(1/3)*b + -(1/3)*c -> like this we have 32 numbers
# dloss/da = -1/3 ~= -1/n

# d_probs
# chain rule here && log(probs) is an outcome of log_probs so log(X) = 1/X
# dloss/dprobs = dloss/dlogprobs * dlogprobs/dprobs = d_log_probs * d(probs.log())/dprobs = d_log_probs * 1.0/probs

# d_counts_sum_inv
# probs = counts * counts_sum_inv
# counts.shape, counts_sum_inv.shape => (torch.Size([32, 27]), torch.Size([32, 1]))
# here it does the Broadcasting Semantics, it will do two operations, first duplicate that column into 27 columns then multiply
# now if we are trying to find c = a * b, d_c/d_b = a, right like other * out.grad
# so here for d_counts_sum_inv = counts * d_probs, but here (32, 27) * (32, 27) => (32, 27) but counts_sum_inv is (32, 1)
# here what it does is in Broadcasting Semantics, it first duplicates the same element multiple times over the column right, yep thats the key we have to see, in Gradient Descent we know if a element is repeating over multiple calculation what we need to do is, we have to cummulative the gradient, thats what we did in MicroGrad `t.grad += (gradient)`
# a11*b1 a12*b1 a13*b1  so here to get b1.grad => a11*d_log_probs[1,1] + a12*d_log_probs[1, 2] + a13*d_log_probs[1, 3]
# a21*b2 a22*b2 a23*b2 # a31*b3 a32*b2 a33*b3

# d_counts
# probs = counts * counts_sum_inv
# d_counts = counts_sum_inv * d_probs
# counts_sum_inv.shape, d_probs.shape => (32, 1) * (32, 27) => Broadcasting Semantics => (32, 27) as result
# It is fine as counts.shape => (32, 27)

# d_counts_sum
# counts_sum_inv = counts_sum ** -1
# d(x^-1) => -1 * x^(-2) => -1 * counts_sum^(-2) * d_counts_sum_inv 

# d_counts
# counts_sum = counts.sum(1, keepdim=True)
# here counts.shape (32, 27) is collapsing into (32, 1)
# here what's happening inside is a11 a12 a13 => b1 (a11 + a12 + a13)
# Now we get the gradient like local.grad * out.grad right, here out.grad is d_counts_sum we already got
# now about local.grad, b1 is (a11 + a12 + a13) so because it is an addition the grad of b1 is going to become out.grad d(x)/dx = 1 evenly for all three elements (a11, a12, a13)
# so local.grad = 1 => 1 * out.grad = out.grad, but it only comes like (32, 1), we have to duplicate this one into 27 more rows.

# d_norm_logits
# counts = norm_logits.exp()
# this is e^x so d(s^x)/dx = e^x, so e^x * out.grad (d_counts)

# d_logit_maxes
# norm_logits = logits - logits_maxes => (32, 27) - (32, 1)
# here Broadcasting Semantics taken place c11 c12 c12 = (a11 a12 a13) - b1
# d(c11)/da11 = 1 (logits) && d(c11)/db1 = -1 (logits_maxes)
# right now we are calculating for logits_maxes, so its -1 * out.grad (d_norm_logits) => (32, 27) => accumulate these to get (32, 1 )

# d_logits
# norm_logits = logits - logits_maxes
# here the same explanation above, but here 1 * out.grad, we do not have to do any accumulation cause the output will be (32, 27)
# AND 
# we have another calculation to lookup to for logits
# logits_maxes = logits.max(1, keepdim=True).values
# here we only need to update the maximum values of logits and the rest will be 0. * d_logits_maxes

# d_h
# logits = h @ W2 + B2
# here to get d_logits/dhdh = W2 => dloss/dh = dloss/dlogits * dlogits/dh = d_logits * W2
# here we can take pen and paper and solve like l11 = a11 * b11 + a12 * b21 + a13 * b31 + c11, like this and try to get everything, watch video if you want
# or we can be a smart ass and get the work done nicely
# here we need d_h, size of it would be (32, 64) same as h right, now to get d_h we know d_logits * W2 => (32, 27) * (64, 27)
# with these two sizes we need to get a size of (32, 64) matrix, to get that (32, 27) @ (64, 27)^T => (32, 27).

# x = a @ b + c => d_a = x @ b^T && d_b = a^T @ x

# d_W2
# logits = h @ W2 + B2 => d_W2 = h * d_logits
# Same logic as above we need (64, 27) size from (32, 64) * (32, 27) => (32, 64)^T @ (32, 27) => (64, 27)

# d_B2
# logits = h @ W2 + B2
# here B2 the size we need is (27), d_B2 = 1 * d_logits => (32, 27).sum(dim=0) => (27)

# d_hpre_act
# h = torch.tanh(hpre_act)
# hpre_act = (1 - tanh(x)^2) * d_h

# I think you got the idea from here, there is not much difficult concept from here

## Exercise 1

In [11]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------
# We have to go through the right side of the Formulas, not the naming side like log_probs, probs, counts_sum_inv, counts_sum, counts.
# rather we have to go like in formulas side like log_probs, probs, counts_sum_inv, counts, counts_sum, counts. (right way).

d_log_probs = torch.zeros_like(log_probs) # (32, 27)
d_log_probs[range(n), yb] = -1.0/n

d_probs = (1.0/probs) * d_log_probs 

d_counts_sum_inv = (counts * d_probs).sum(dim=1, keepdim=True)
 
d_counts =  counts_sum_inv * d_probs # this won't become True, because counts is being used multiple times above, so we need to accumulate the gradients of these counts

d_counts_sum =  -1 * counts_sum**(-2) * d_counts_sum_inv

d_counts += torch.ones_like(counts) * d_counts_sum # we need to cummulative this one.

d_norm_logits =  norm_logits.exp() * d_counts # counts * d_counts

d_logits_maxes = (-1 * d_norm_logits).sum(dim=1, keepdim=True)

d_logits = (1 * d_norm_logits) # d_norm_logits.clone()
d_logits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * d_logits_maxes

d_h = d_logits @ W2.T

d_W2 = h.T @ d_logits

d_B2 = (1 * d_logits).sum(dim=0)

d_hpre_act = (1 - h*h) * d_h

d_bn_gain = (bn_raw * d_hpre_act).sum(dim=0, keepdim=True)

d_bn_bias = (1 * d_hpre_act).sum(dim=0, keepdim=True)

d_bn_raw = (bn_gain * d_hpre_act)

d_bn_var_inv = (bn_diff * d_bn_raw).sum(dim=0, keepdim=True)

d_bn_diff = bn_var_inv * d_bn_raw

d_bn_var = (-0.5 * (bn_var + 1e-5)**(-1.5)) * d_bn_var_inv

d_bn_diff_2 = (1.0/(n-1)) * torch.ones_like(bn_diff_2) * d_bn_var

d_bn_diff += 2 * bn_diff * d_bn_diff_2 

d_bn_mean_i = (-d_bn_diff).sum(dim=0, keepdim=True)

d_hpre_bn = 1 * d_bn_diff # d_bn_diff.clone()
d_hpre_bn += (1.0/n) * torch.ones_like(hpre_bn) * d_bn_mean_i

d_emb_cat = d_hpre_bn @ W1.T

d_W1 = emb_cat.T @ d_hpre_bn

d_B1 = d_hpre_bn.sum(dim=0)

d_emb = d_emb_cat.view(emb.shape)

d_C = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[i, j]
        d_C[ix] += d_emb[i, j]


cmp('log_probs', d_log_probs, log_probs)
cmp('probs', d_probs, probs)
cmp('counts_sum_inv', d_counts_sum_inv, counts_sum_inv)
cmp('counts_sum', d_counts_sum, counts_sum)
cmp('counts', d_counts, counts)
cmp('norm_logits', d_norm_logits, norm_logits)
cmp('logit_maxes', d_logits_maxes, logits_maxes)
cmp('logits', d_logits, logits)
cmp('h', d_h, h)
cmp('W2', d_W2, W2)
cmp('b2', d_B2, B2)
cmp('hpreact', d_hpre_act, hpre_act)
cmp('bngain', d_bn_gain, bn_gain)
cmp('bnbias', d_bn_bias, bn_bias)
cmp('bnraw', d_bn_raw, bn_raw)
cmp('bnvar_inv', d_bn_var_inv, bn_var_inv)
cmp('bnvar', d_bn_var, bn_var)
cmp('bndiff2', d_bn_diff_2, bn_diff_2)
cmp('bndiff', d_bn_diff, bn_diff)
cmp('bnmeani', d_bn_mean_i, bn_mean_i)
cmp('hprebn', d_hpre_bn, hpre_bn)
cmp('embcat', d_emb_cat, emb_cat)
cmp('W1', d_W1, W1)
cmp('b1', d_B1, B1)
cmp('emb', d_emb, emb)
cmp('C', d_C, C) 

log_probs       | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approximate: True  | maxdiff: 2.7939677238464355e-09
bnbias          | exact: False | approximate: True  | maxdiff: 4.6566128730773926e-09
bnraw 

exact: True                                         Bit-for-bit identical ✅  
exact: False + approximate: True + maxdiff ~1e-9    Correct ✅ just float32 rounding  
exact: False + approximate: False + maxdiff ~1e-7   Actual bug ❌  

## Exercise 2

In [12]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.338137626647949 diff: -4.76837158203125e-07


In [13]:
# ! '/home/mrx/Pictures/Screenshots/Screenshot From 2026-03-21 00-43-28.png'

# backward pass

# -----------------
# YOUR CODE HERE :)
# -----------------
d_logits = F.softmax(logits, dim=1)
d_logits[range(n), yb] -= 1
d_logits /= n # We do Mean right so 1/n * local.grad

cmp('logits', d_logits, logits) # I can only get approximate to be true  

logits          | exact: False | approximate: True  | maxdiff: 7.2177499532699585e-09


## Exercise 3

In [14]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpre_act_fast = bn_gain * (hpre_bn - hpre_bn.mean(0, keepdim=True)) / torch.sqrt(hpre_bn.var(0, keepdim=True, unbiased=True) + 1e-5) + bn_bias
print('max diff:', (hpre_act_fast - hpre_act).abs().max()) 

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [15]:
# ! '/home/mrx/Pictures/Screenshots/Screenshot From 2026-03-21 01-07-10.png'

# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
# -----------------
d_hpre_bn = bn_gain * bn_var_inv/n * (n * d_hpre_act - d_hpre_act.sum(dim=0) - n/(n-1) * bn_raw * (d_hpre_act * bn_raw).sum(dim=0))

cmp('hprebn', d_hpre_bn, hpre_bn) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


## Exercise 4

In [35]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass
# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bn_gain = torch.randn((1, n_hidden))*0.1 + 1.0
bn_bias = torch.randn((1, n_hidden))*0.1
parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True
# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []
# use this context manager for efficiency once your backward pass is written (TODO)
with torch.no_grad(): # <-- keep commented out so PyTorch can compute grads for comparison
# kick off optimization
  for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    Xb, Yb = X_train[ix], y_train[ix] # batch X,Y
    # forward pass
    emb = C[Xb] # embed the characters into vectors
    emb_cat = emb.view(emb.shape[0], -1) # concatenate the vectors
    # Linear layer
    hpre_bn = emb_cat @ W1 + b1 # hidden layer pre-activation
    # BatchNorm layer
    # -------------------------------------------------------------
    bn_mean = hpre_bn.mean(0, keepdim=True)
    bn_var = hpre_bn.var(0, keepdim=True, unbiased=True)
    bn_var_inv = (bn_var + 1e-5)**-0.5
    bn_raw = (hpre_bn - bn_mean) * bn_var_inv
    hpre_act = bn_gain * bn_raw + bn_bias
    # -------------------------------------------------------------
    # Non-linearity
    h = torch.tanh(hpre_act) # hidden layer
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function
    # backward pass
    for p in parameters:
      p.grad = None
    # loss.backward() # use this for correctness comparisons, delete it later!
    # manual backprop! #swole_doge_meme
    # -----------------
    # YOUR CODE HERE :)
    d_logits = F.softmax(logits, dim=1)
    d_logits[range(n), Yb] -= 1
    d_logits /= n
    # 2nd layer backprop
    d_h = d_logits @ W2.T
    d_W2 = h.T @ d_logits
    d_b2 = d_logits.sum(0)
    # tanh
    d_hpre_act = (1.0 - h**2) * d_h
    # BatchNorm backprop
    d_bn_gain = (bn_raw * d_hpre_act).sum(0, keepdim=True)
    d_bn_bias = d_hpre_act.sum(0, keepdim=True)
    d_hpre_bn = bn_gain * bn_var_inv/n * (n * d_hpre_act - d_hpre_act.sum(dim=0) - n/(n-1) * bn_raw * (d_hpre_act * bn_raw).sum(dim=0))
    # 1st layer backprop
    d_emb_cat = d_hpre_bn @ W1.T
    d_W1 = emb_cat.T @ d_hpre_bn
    d_b1 = d_hpre_bn.sum(0)
    # Embedding
    d_emb = d_emb_cat.view(emb.shape)
    d_C = torch.zeros_like(C)
    for ii in range(Xb.shape[0]):
      for jj in range(Xb.shape[1]):
        ix = Xb[ii, jj]
        d_C[ix] += d_emb[ii, jj]
    grads = [d_C, d_W1, d_b1, d_W2, d_b2, d_bn_gain, d_bn_bias]
    # -----------------
    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p, grad in zip(parameters, grads):
      # p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
      p.data += -lr * grad # new way of swole doge TODO: enable
    # track stats
    if i % 10000 == 0: # print every once in a while
      print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())
    # if i >= 100: # TODO: delete early breaking when you're ready to train the full net
    #   break

12297
      0/ 200000: 3.7788
  10000/ 200000: 2.1708
  20000/ 200000: 2.4212
  30000/ 200000: 2.4838
  40000/ 200000: 1.9324
  50000/ 200000: 2.3811
  60000/ 200000: 2.3741
  70000/ 200000: 2.0478
  80000/ 200000: 2.3515
  90000/ 200000: 2.1695
 100000/ 200000: 1.9174
 110000/ 200000: 2.2678
 120000/ 200000: 1.9830
 130000/ 200000: 2.4510
 140000/ 200000: 2.3439
 150000/ 200000: 2.1758
 160000/ 200000: 1.8727
 170000/ 200000: 1.8077
 180000/ 200000: 1.9635
 190000/ 200000: 1.8750


In [33]:
# useful for checking your gradients
# Just run the above code with no_grad and in for loop comment out .backward & update p.grad, now with our manual backward and our grads updation, run the epochs fully
for p,g in zip(parameters, grads):
  cmp(str(tuple(p.shape)), g, p)

(27, 10)        | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
(30, 200)       | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
(200,)          | exact: False | approximate: True  | maxdiff: 4.423782229423523e-09
(200, 27)       | exact: False | approximate: True  | maxdiff: 2.2351741790771484e-08
(27,)           | exact: False | approximate: True  | maxdiff: 7.450580596923828e-09
(1, 200)        | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
(1, 200)        | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09


In [40]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[X_train]
  emb_cat = emb.view(emb.shape[0], -1)
  hpre_act = emb_cat @ W1 + b1
  # measure the mean/std over the entire training set
  bn_mean = hpre_act.mean(0, keepdim=True)
  bn_var = hpre_act.var(0, keepdim=True, unbiased=True)


In [45]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (X_train, y_train),
    'val': (X_valid, y_valid),
    'test': (X_test, y_test),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  emb_cat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpre_act = emb_cat @ W1 + b1
  hpre_act = bn_gain * (hpre_act - bn_mean) * (bn_var + 1e-5)**-0.5 + bn_bias
  h = torch.tanh(hpre_act) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.071136474609375
val 2.1124188899993896


In [42]:
# Andrej achieved:
# train 2.0718822479248047
# val 2.1162495613098145

# I Achieved
# train 2.071136474609375
# val 2.1124188899993896

In [48]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bn_gain * (hpreact - bn_mean) * (bn_var + 1e-5)**-0.5 + bn_bias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))

carmahzamille$
khylin$
khelly$
sacarson$
mahnen$
deliah$
jareei$
nellara$
chaiiv$
kaleigh$
ham$
join$
quint$
shoilea$
jadiquinterri$
jaryxia$
kaellinslee$
dae$
iia$
gian$
